# Modeles lineaires accidentologie (3 variations hyperparametres)

Objectif:
- entrainer 3 modeles lineaires (Ridge, Lasso, ElasticNet)
- faire varier les hyperparametres via recherche
- comparer les resultats et garder le meilleur modele
- preparer les artefacts pour une future etape MLflow

Important: les commandes MLflow sont integrees mais desactivees (non executees).

In [ ]:
from __future__ import annotations

import json
from datetime import UTC, datetime
from pathlib import Path

import joblib
import numpy as np
import pandas as pd
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.linear_model import ElasticNet, Lasso, Ridge
from sklearn.metrics import (
    accuracy_score,
    average_precision_score,
    f1_score,
    mean_absolute_error,
    mean_squared_error,
    precision_score,
    r2_score,
    recall_score,
    roc_auc_score,
)
from sklearn.model_selection import (
    RandomizedSearchCV,
    StratifiedKFold,
    train_test_split,
)
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler


def find_project_root(marker: str = "out") -> Path:
    for p in [Path.cwd(), *Path.cwd().parents]:
        if (p / marker).exists():
            return p
    return Path.cwd()


ROOT = find_project_root("out")
DATA_PATH = ROOT / "out" / "filtered" / "accidents_model_ready_kept.csv"
ARTIFACT_DIR = ROOT / "out" / "linear_regression_experiments"
ARTIFACT_DIR.mkdir(parents=True, exist_ok=True)

TARGET = "grave"
SEED = 42
CV_SPLITS = 3
N_ITER = 20
THRESHOLD = 0.5
MAX_ROWS = None  # ex: 50000 pour accelerer

print("ROOT:", ROOT)
print("DATA_PATH:", DATA_PATH)
print("ARTIFACT_DIR:", ARTIFACT_DIR)

In [ ]:
assert DATA_PATH.exists(), f"Fichier introuvable: {DATA_PATH}"

df = pd.read_csv(DATA_PATH, sep=";", low_memory=False, nrows=MAX_ROWS)
assert TARGET in df.columns, f"Colonne cible absente: {TARGET}"

df = df.dropna(subset=[TARGET]).copy()
df[TARGET] = pd.to_numeric(df[TARGET], errors="coerce")
df = df.dropna(subset=[TARGET]).copy()
df[TARGET] = df[TARGET].astype(int)

X = df.drop(columns=[TARGET]).copy()
y = df[TARGET].copy()

print("Shape:", df.shape)
print("Nb features:", X.shape[1])
print("Taux grave=1:", round(float(y.mean()), 4))

## Preprocessing

- split train/test stratifie
- imputation numerique et categorielle
- one-hot encoding pour les colonnes categorielles
- standardisation des variables numeriques

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=SEED,
    stratify=y,
)

cat_cols = X_train.select_dtypes(
    include=["object", "category", "string"]
).columns.tolist()
num_cols = [c for c in X_train.columns if c not in cat_cols]

num_pipe = Pipeline(
    steps=[
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler", StandardScaler()),
    ]
)

cat_pipe = Pipeline(
    steps=[
        ("imputer", SimpleImputer(strategy="most_frequent")),
        ("onehot", OneHotEncoder(handle_unknown="ignore", sparse_output=True)),
    ]
)

preprocess = ColumnTransformer(
    transformers=[
        ("num", num_pipe, num_cols),
        ("cat", cat_pipe, cat_cols),
    ],
    remainder="drop",
)

print("Train:", X_train.shape, "| Test:", X_test.shape)
print("Numeriques:", len(num_cols), "| Categorielles:", len(cat_cols))

## 3 variations hyperparametres

Experiences:
1. `ridge_opt`
2. `lasso_opt`
3. `elasticnet_opt`

Chaque experience optimise `neg_root_mean_squared_error` via CV.

In [ ]:
def clip_to_unit_interval(x: np.ndarray) -> np.ndarray:
    return np.clip(x, 0.0, 1.0)


def evaluate_predictions(
    y_true: pd.Series, y_score: np.ndarray, threshold: float = THRESHOLD
) -> dict[str, float]:
    y_prob = clip_to_unit_interval(y_score)
    y_pred = (y_prob >= threshold).astype(int)

    return {
        "threshold": float(threshold),
        "rmse": float(mean_squared_error(y_true, y_score, squared=False)),
        "mae": float(mean_absolute_error(y_true, y_score)),
        "r2": float(r2_score(y_true, y_score)),
        "accuracy": float(accuracy_score(y_true, y_pred)),
        "precision": float(precision_score(y_true, y_pred, zero_division=0)),
        "recall": float(recall_score(y_true, y_pred, zero_division=0)),
        "f1": float(f1_score(y_true, y_pred, zero_division=0)),
        "roc_auc": float(roc_auc_score(y_true, y_prob)),
        "pr_auc": float(average_precision_score(y_true, y_prob)),
    }


def make_search(model, param_dist: dict) -> RandomizedSearchCV:
    pipe = Pipeline(
        steps=[
            ("prep", preprocess),
            ("model", model),
        ]
    )
    cv = StratifiedKFold(n_splits=CV_SPLITS, shuffle=True, random_state=SEED)
    return RandomizedSearchCV(
        estimator=pipe,
        param_distributions=param_dist,
        n_iter=N_ITER,
        scoring="neg_root_mean_squared_error",
        cv=cv,
        n_jobs=-1,
        random_state=SEED,
        verbose=1,
        refit=True,
    )


EXPERIMENTS = [
    {
        "run_name": "ridge_opt",
        "model": Ridge(random_state=SEED),
        "param_dist": {
            "model__alpha": [0.001, 0.01, 0.1, 1, 3, 10, 30, 100],
            "model__fit_intercept": [True, False],
            "model__solver": ["auto", "lsqr", "sag", "sparse_cg"],
        },
    },
    {
        "run_name": "lasso_opt",
        "model": Lasso(random_state=SEED, max_iter=20000),
        "param_dist": {
            "model__alpha": [1e-5, 3e-5, 1e-4, 3e-4, 1e-3, 3e-3, 1e-2, 3e-2],
            "model__fit_intercept": [True, False],
            "model__selection": ["cyclic", "random"],
        },
    },
    {
        "run_name": "elasticnet_opt",
        "model": ElasticNet(random_state=SEED, max_iter=25000),
        "param_dist": {
            "model__alpha": [1e-5, 3e-5, 1e-4, 3e-4, 1e-3, 3e-3, 1e-2, 3e-2],
            "model__l1_ratio": [0.1, 0.25, 0.4, 0.6, 0.75, 0.9],
            "model__fit_intercept": [True, False],
            "model__selection": ["cyclic", "random"],
        },
    },
]

In [ ]:
trained = {}
rows = []

for exp in EXPERIMENTS:
    run_name = exp["run_name"]
    print(f"\n=== {run_name} ===")

    search = make_search(exp["model"], exp["param_dist"])
    search.fit(X_train, y_train)

    best_model = search.best_estimator_
    y_score_test = best_model.predict(X_test)
    metrics = evaluate_predictions(y_test, y_score_test, threshold=THRESHOLD)

    trained[run_name] = {
        "search": search,
        "model": best_model,
        "y_score_test": y_score_test,
        "metrics": metrics,
    }

    rows.append(
        {
            "run_name": run_name,
            "cv_best_neg_rmse": float(search.best_score_),
            **metrics,
            "best_params": search.best_params_,
        }
    )

results_df = (
    pd.DataFrame(rows)
    .sort_values(["f1", "roc_auc"], ascending=False)
    .reset_index(drop=True)
)
results_df[
    [
        "run_name",
        "cv_best_neg_rmse",
        "rmse",
        "mae",
        "r2",
        "accuracy",
        "precision",
        "recall",
        "f1",
        "roc_auc",
        "pr_auc",
    ]
]

In [ ]:
registry_rows = []

for row in rows:
    run_name = row["run_name"]
    model = trained[run_name]["model"]
    y_score_test = trained[run_name]["y_score_test"]

    model_path = ARTIFACT_DIR / f"{run_name}.joblib"
    pred_path = ARTIFACT_DIR / f"{run_name}_predictions.csv"
    meta_path = ARTIFACT_DIR / f"{run_name}_meta.json"

    joblib.dump(model, model_path)

    pred_df = pd.DataFrame(
        {
            "y_true": y_test.to_numpy(),
            "y_score": y_score_test,
            "y_prob_clipped": clip_to_unit_interval(y_score_test),
            "pred_05": (clip_to_unit_interval(y_score_test) >= THRESHOLD).astype(int),
        }
    )
    pred_df.to_csv(pred_path, index=False)

    meta = {
        "run_name": run_name,
        "dataset": str(DATA_PATH),
        "target": TARGET,
        "threshold": THRESHOLD,
        "seed": SEED,
        "cv_splits": CV_SPLITS,
        "n_iter": N_ITER,
        "cv_best_neg_rmse": row["cv_best_neg_rmse"],
        "best_params": trained[run_name]["search"].best_params_,
        "metrics_test": trained[run_name]["metrics"],
        "model_path": str(model_path),
        "predictions_path": str(pred_path),
        "created_at_utc": datetime.now(UTC).isoformat(),
    }

    with open(meta_path, "w", encoding="utf-8") as f:
        json.dump(meta, f, ensure_ascii=False, indent=2)

    registry_rows.append(
        {
            "run_name": run_name,
            "model_path": str(model_path),
            "predictions_path": str(pred_path),
            "meta_path": str(meta_path),
        }
    )

registry_df = pd.DataFrame(registry_rows)
mlflow_ready_df = results_df.merge(registry_df, on="run_name", how="left")
mlflow_ready_df

In [ ]:
best_idx = results_df["f1"].idxmax()
best_run = results_df.loc[best_idx, "run_name"]

print("Best run (selon F1):", best_run)
print("Best params:")
print(results_df.loc[best_idx, "best_params"])

## Integration MLflow (desactivee)

Les commandes ci-dessous sont prêtes pour la prochaine etape.
Par defaut, `ENABLE_MLFLOW = False`, donc rien ne se lance.

In [ ]:
ENABLE_MLFLOW = False
MLFLOW_TRACKING_URI = "http://127.0.0.1:5000"
MLFLOW_EXPERIMENT = "accidentologie_model_benchmark"

if ENABLE_MLFLOW:
    import mlflow
    import mlflow.sklearn
    import numpy as np

    # Active autolog des params/metrics tout en gardant le log_model manuel ci-dessous.
    mlflow.sklearn.autolog(log_models=False, silent=True)

    def _to_params(d):
        out = {}
        if isinstance(d, dict):
            for k, v in d.items():
                if isinstance(
                    v, (int, float, str, bool, np.integer, np.floating, np.bool_)
                ):
                    out[k] = v.item() if hasattr(v, "item") else v
        return out

    def _split_metrics(d):
        cls_keys = {
            "accuracy",
            "precision",
            "recall",
            "f1",
            "roc_auc",
            "pr_auc",
            "threshold",
        }
        reg_keys = {"rmse", "mae", "r2"}
        cls = {}
        reg = {}
        if isinstance(d, dict):
            for k, v in d.items():
                if not isinstance(v, (int, float, np.integer, np.floating)):
                    continue
                if k in cls_keys:
                    cls[f"test_{k}"] = float(v)
                elif k in reg_keys:
                    reg[f"reg_test_{k}"] = float(v)
        return cls, reg

    mlflow.set_tracking_uri(MLFLOW_TRACKING_URI)
    mlflow.set_experiment(MLFLOW_EXPERIMENT)

    for row in rows:
        run_name = row["run_name"]
        run_payload = trained.get(run_name, {})

        search = run_payload.get("search")
        model = run_payload.get("model")
        if search is None or model is None:
            print(f"[mlflow] skip {run_name}: search/model manquant")
            continue

        cls_metrics, reg_metrics = _split_metrics(run_payload.get("metrics"))

        model_path = ARTIFACT_DIR / f"{run_name}.joblib"
        pred_path = ARTIFACT_DIR / f"{run_name}_predictions.csv"
        meta_path = ARTIFACT_DIR / f"{run_name}_meta.json"

        with mlflow.start_run(run_name=run_name):
            mlflow.set_tags(
                {
                    "notebook": "13_linear_regression_mlflow_prep.ipynb",
                    "model_family": "linear",
                    "model_flavor": "sklearn",
                    "tag": "accidentologie",
                }
            )

            mlflow.log_param("optimized_for", "neg_root_mean_squared_error")
            mlflow.log_param("cv_primary_metric", "neg_root_mean_squared_error")
            mlflow.log_metric("cv_primary_score", float(search.best_score_))
            mlflow.log_params(_to_params(search.best_params_))

            if cls_metrics:
                mlflow.log_metrics(cls_metrics)
            if reg_metrics:
                mlflow.log_metrics(reg_metrics)

            mlflow.sklearn.log_model(model, artifact_path="model")

            if model_path.exists():
                mlflow.log_artifact(str(model_path), artifact_path="models")
            if pred_path.exists():
                mlflow.log_artifact(str(pred_path), artifact_path="predictions")
            if meta_path.exists():
                mlflow.log_artifact(str(meta_path), artifact_path="metadata")
else:
    print("MLflow desactive. Passe ENABLE_MLFLOW=True a la prochaine etape.")